# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/toBESkiii/FlyRank_AI_Intership/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*


My lane is **Refresh / Content Opportunity Scoring**. The purpose of the project is to help an SEO specialist or content editor decide which pages should be investigated first.

I will start with **Logistic Regression** because my temporary proxy target is binary: a page is either labelled as declining or not declining. Logistic Regression also produces a probability score, which can be used to rank pages from highest to lowest review priority.

I will then compare it with a **Random Forest**. Random Forest can learn more complex and non-linear relationships between page signals that Logistic Regression may miss. However, the more complex model will only be considered better if it produces a meaningful improvement on the same held-out data.

The final output is not simply a yes-or-no prediction. Each model's predicted probability will be used as a ranking score so that the highest-priority pages can be reviewed first.

My primary metric will be **Precision@10**, matching the top-10 review used for my Week-4 baseline. I will also report Precision@50 to check whether the result remains useful deeper in the ranked queue.

The temporary proxy target is based on whether `trend_direction` equals `down`. This is an observed teaching proxy rather than a true future outcome.

The columns `trend_direction` and `trend_pct` will not be used as model features because they directly define the proxy target and would cause target leakage.


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

# Load the same starter dataset used for the Week-4 baseline
DATA_URL = (
    "https://raw.githubusercontent.com/"
    "toBESkiii/FlyRank_AI_Intership/main/"
    "data/raw/content_refresh_anonymized.csv"
)

page_dataset = pd.read_csv(DATA_URL)

# Create the temporary decline proxy
# 1 = observed downward trend
# 0 = any other trend category
page_dataset["decline_proxy"] = (
    page_dataset["trend_direction"] == "down"
).astype(int)

print("Dataset loaded successfully.")
print("Pages:", len(page_dataset))
print("Columns:", page_dataset.shape[1])
print("Clients:", page_dataset["client_id"].nunique())
print(
    "Decline proxy rate:",
    round(page_dataset["decline_proxy"].mean(), 3)
)


Dataset loaded successfully.
Pages: 30000
Columns: 45
Clients: 32
Decline proxy rate: 0.542


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I will use a **grouped train/test split based on `client_id`**.

Pages belonging to the same client will remain entirely within either the training set or the test set. This prevents pages from the same client appearing on both sides of the split.

A random page-level split could make the evaluation overly optimistic because pages from one client may share similar search, content or performance characteristics. The model could learn client-specific patterns during training and then encounter similar pages from the same client during testing.

Using a client-level holdout provides a stronger test of whether the model can generalise its ranking to previously unseen clients.

Approximately 80% of clients will be used for training and 20% will be held out for testing.

The Week-4 baseline and both machine-learning models will be evaluated on this exact same test set using the same metrics so that the comparison is fair.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

# ---------------------------------------------------------
# Create a grouped train/test split
# ---------------------------------------------------------

grouped_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_indices, test_indices = next(
    grouped_split.split(
        page_dataset,
        y=page_dataset["decline_proxy"],
        groups=page_dataset["client_id"]
    )
)

training_dataset = (
    page_dataset
    .iloc[train_indices]
    .copy()
)

test_dataset = (
    page_dataset
    .iloc[test_indices]
    .copy()
)

# ---------------------------------------------------------
# Check the split
# ---------------------------------------------------------

training_clients = set(training_dataset["client_id"])
test_clients = set(test_dataset["client_id"])

shared_clients = training_clients.intersection(test_clients)

print("Training pages:", len(training_dataset))
print("Test pages:", len(test_dataset))

print("\nTraining clients:", len(training_clients))
print("Test clients:", len(test_clients))

print("\nClients appearing in both sets:", len(shared_clients))

print(
    "Training decline rate:",
    round(training_dataset["decline_proxy"].mean(), 3)
)

print(
    "Test decline rate:",
    round(test_dataset["decline_proxy"].mean(), 3)
)

Training pages: 23837
Test pages: 6163

Training clients: 25
Test clients: 7

Clients appearing in both sets: 0
Training decline rate: 0.55
Test decline rate: 0.511


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*


I will compare three approaches on the exact same held-out test set:

1. My Week-4 fixed-rule baseline
2. Logistic Regression
3. Random Forest

The baseline rule remains:

> Prioritise pages that have not been updated for at least 90 days and have received at least 300 impressions during the previous 90 days.

For the machine-learning models, I will use five leakage-safe page signals:

* `days_since_last_update`
* `impressions_90d`
* `ctr`
* `avg_position`
* `search_volume`

Missing values will be handled using values learned from the training set only. `avg_position = 0` will be treated as missing because zero represents unavailable position information rather than a genuine search position.

Both classifiers will produce a probability of the decline proxy. These probabilities will be used as ranking scores.

The baseline and both models will then be evaluated on the same held-out client set using **Precision@10** and **Precision@50**. This ensures that any improvement comes from the method rather than from using different test data.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# ---------------------------------------------------------
# 1. Define leakage-safe features and target
# ---------------------------------------------------------

model_features = [
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "avg_position",
    "search_volume"
]

X_train = training_dataset[model_features].copy()
X_test = test_dataset[model_features].copy()

y_train = training_dataset["decline_proxy"].copy()
y_test = test_dataset["decline_proxy"].copy()

# In this dataset, avg_position = 0 means position data is unavailable.
# Treat it as missing rather than as a genuine search position.
X_train["avg_position"] = X_train["avg_position"].replace(0, np.nan)
X_test["avg_position"] = X_test["avg_position"].replace(0, np.nan)

print("Training feature shape:", X_train.shape)
print("Test feature shape:", X_test.shape)


# ---------------------------------------------------------
# 2. Logistic Regression
# ---------------------------------------------------------

logistic_model = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

logistic_model.fit(X_train, y_train)

logistic_scores = logistic_model.predict_proba(X_test)[:, 1]


# ---------------------------------------------------------
# 3. Random Forest
# ---------------------------------------------------------

random_forest_model = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=8,
                min_samples_leaf=10,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

random_forest_model.fit(X_train, y_train)

random_forest_scores = (
    random_forest_model.predict_proba(X_test)[:, 1]
)


# ---------------------------------------------------------
# 4. Re-create the Week-4 baseline on SAME test set
# ---------------------------------------------------------

baseline_test = test_dataset.copy()

baseline_test["baseline_qualifies"] = (
    (baseline_test["days_since_last_update"] >= 90)
    & (baseline_test["impressions_90d"] >= 300)
)

baseline_test["baseline_score"] = np.where(
    baseline_test["baseline_qualifies"],
    baseline_test["impressions_90d"],
    0
)

baseline_scores = baseline_test["baseline_score"].to_numpy()


# ---------------------------------------------------------
# 5. Precision@K function
# ---------------------------------------------------------

def precision_at_k(y_true, ranking_scores, k):
    """
    Calculate the proportion of positive target rows
    among the K highest-ranked scores.
    """

    y_true_array = np.asarray(y_true)
    ranking_scores_array = np.asarray(ranking_scores)

    actual_k = min(k, len(y_true_array))

    top_indices = np.argsort(
        ranking_scores_array
    )[::-1][:actual_k]

    return y_true_array[top_indices].mean()


# ---------------------------------------------------------
# 6. Evaluate all three methods
# ---------------------------------------------------------

results = []

methods = {
    "Week-4 Baseline": baseline_scores,
    "Logistic Regression": logistic_scores,
    "Random Forest": random_forest_scores
}

for method_name, scores in methods.items():

    results.append(
        {
            "Method": method_name,
            "Precision@10": precision_at_k(
                y_test,
                scores,
                10
            ),
            "Precision@50": precision_at_k(
                y_test,
                scores,
                50
            )
        }
    )

model_comparison = pd.DataFrame(results)

model_comparison[
    ["Precision@10", "Precision@50"]
] = model_comparison[
    ["Precision@10", "Precision@50"]
].round(3)

print(
    "Baseline-qualified pages in test set:",
    int(baseline_test["baseline_qualifies"].sum())
)

print("\nMODEL VS BASELINE")
display(model_comparison)

Training feature shape: (23837, 5)
Test feature shape: (6163, 5)
Baseline-qualified pages in test set: 684

MODEL VS BASELINE


,Method,Precision@10,Precision@50
0,Week-4 Baseline,0.3,0.30
1,Logistic Regression,0.4,0.64
2,Random Forest,0.8,0.78


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Random Forest produced the strongest ranking results, so I will inspect it more closely rather than relying only on its headline Precision@K scores.

I will use permutation importance to examine which input features contribute most to its Precision@50 performance. This works by shuffling one feature at a time and measuring how much the ranking performance changes. A larger performance drop suggests that the model relies more strongly on that feature.

I will also inspect concrete false-positive recommendations near the top of the ranked queue. These are pages that the Random Forest ranked highly even though their observed decline proxy is 0.

This error review is important because even a strong Precision@10 or Precision@50 score does not mean every recommendation is correct.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.inspection import permutation_importance

# ---------------------------------------------------------
# 1. Permutation importance using our ranking metric
# ---------------------------------------------------------

def precision_at_50_scorer(model, features, target):
    """
    Score a fitted model using Precision@50.
    """

    probability_scores = model.predict_proba(features)[:, 1]

    return precision_at_k(
        target,
        probability_scores,
        50
    )


permutation_results = permutation_importance(
    random_forest_model,
    X_test,
    y_test,
    scoring=precision_at_50_scorer,
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

feature_importance = pd.DataFrame(
    {
        "feature": model_features,
        "importance_mean": permutation_results.importances_mean,
        "importance_std": permutation_results.importances_std
    }
)

feature_importance = (
    feature_importance
    .sort_values(
        by="importance_mean",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Random Forest permutation importance:")
display(feature_importance)

Random Forest permutation importance:


,feature,importance_mean,importance_std
0,impressions_90d,0.248,0.042143
1,avg_position,0.200,0.057271
2,ctr,0.078,0.056178
3,search_volume,0.048,0.051536
4,days_since_last_update,0.038,0.073457


In [5]:
# ---------------------------------------------------------
# 2. Build Random Forest ranked test queue
# ---------------------------------------------------------

random_forest_review = test_dataset[
    [
        "content_id",
        "client_id",
        "decline_proxy",
        "days_since_last_update",
        "impressions_90d",
        "ctr",
        "avg_position",
        "search_volume"
    ]
].copy()

random_forest_review["random_forest_score"] = (
    random_forest_scores
)

random_forest_review = (
    random_forest_review
    .sort_values(
        by="random_forest_score",
        ascending=False
    )
    .reset_index(drop=True)
)

random_forest_review.insert(
    0,
    "review_rank",
    range(1, len(random_forest_review) + 1)
)


# ---------------------------------------------------------
# 3. Inspect errors inside top 50
# ---------------------------------------------------------

top_50_random_forest = (
    random_forest_review
    .head(50)
    .copy()
)

correct_top_50 = (
    top_50_random_forest["decline_proxy"] == 1
).sum()

false_positive_top_50 = (
    top_50_random_forest["decline_proxy"] == 0
).sum()

print(
    "Correct decline-proxy pages in Top 50:",
    correct_top_50
)

print(
    "False positives in Top 50:",
    false_positive_top_50
)


# ---------------------------------------------------------
# 4. Show three concrete wrong recommendations
# ---------------------------------------------------------

top_false_positives = (
    top_50_random_forest[
        top_50_random_forest["decline_proxy"] == 0
    ]
    .head(3)
)

print("\nThree high-ranked false positives:")

display(
    top_false_positives[
        [
            "review_rank",
            "content_id",
            "random_forest_score",
            "days_since_last_update",
            "impressions_90d",
            "ctr",
            "avg_position",
            "search_volume",
            "decline_proxy"
        ]
    ]
)

Correct decline-proxy pages in Top 50: 39
False positives in Top 50: 11

Three high-ranked false positives:


,review_rank,content_id,random_forest_score,days_since_last_update,impressions_90d,ctr,avg_position,search_volume,decline_proxy
0,1,content_26d48a980581,0.799401,106,1266,0.00,4.6,0.0,0
4,5,content_50cd61b8e18d,0.785401,14,3799,0.05,19.1,0.0,0
12,13,content_aea979e1b10c,0.770008,20,1677,0.06,21.9,0.0,0


### Interpretation of model behaviour and errors

Random Forest produced the strongest ranking performance, with 39 of its top 50 recommendations matching the decline proxy. This gives a Precision@50 of 0.78. However, 11 of the top 50 recommendations were false positives, showing that the model is useful but still imperfect.

Permutation importance showed that `impressions_90d` was the strongest feature, with mean importance of 0.248, followed by `avg_position` at 0.200. CTR contributed less at 0.078, while `search_volume` and `days_since_last_update` had relatively small and less stable importance.

This is consistent with my Week-4 signal audit, where staleness received a MIXED verdict. The stronger Random Forest model appears to benefit more from combining search visibility and search-position information than from relying mainly on a fixed staleness threshold.

The highest-ranked false positive was ranked first with a Random Forest score of approximately 0.80. It had 1,266 impressions, a CTR of 0.00 and an average position of 4.6, but its decline proxy was 0. This suggests that the model may sometimes interpret unusual combinations such as strong search visibility with very low click-through behaviour as evidence of decline when no observed downward trend is present.

The other high-ranked false positives at ranks 5 and 13 were only 14 and 20 days since their last update. This further suggests that page age alone is not the main factor driving the Random Forest ranking.

Overall, the model substantially improves the ranked review queue compared with the simple Week-4 baseline, but the false positives show why its recommendations should still be treated as decision support for an SEO specialist rather than automatic refresh decisions.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.